<a href="https://colab.research.google.com/github/alyssanicoletech-cyberstar/Data-Curation/blob/main/LLM_Training_Session_A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initial Installs for Model Fine-Tuning

 1. Transformer Module Library

In [ ]:
!pip install transformers

2. Tokenizer

In [ ]:
!pip install AutoTokenizer

ERROR: Could not find a version that satisfies the requirement AutoTokenizer (from versions: none)
ERROR: No matching distribution found for AutoTokenizer


3. Dataset Library

In [ ]:
!pip install datasets

4. Distributed Training Library

In [ ]:
!pip install accelerate

5. TRL Library

In [ ]:
!pip install git+https://github.com/huggingface/trl.git



  Cloning https://github.com/huggingface/trl.git to /tmp/pip-req-build-50vdriva
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/trl.git /tmp/pip-req-build-50vdriva
  Resolved https://github.com/huggingface/trl.git to commit 95f5c97c5bd7dab813400a0d4ca2a99d1f2f7297
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


If -U trl Library is INSTALLED then IMPORT SFTTrainer

In [ ]:

from trl import SFTTrainer



 5. Parameter Efficient Fine-Tuning

In [ ]:
!pip install peft

Deep Seed Library

In [ ]:
!pip install deepspeed

If Transformers Library is INSTALLED Then IMPORT model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer


If Model is imported then load the model

In [ ]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


IF DATA PIPLELINE INSTALL IS TRUE: IMPORT PIPELINE TO LOAD A DATA SET WHERE THE DATA SET IS SET TO ""

In [ ]:
from datasets import load_dataset

dataset = load_dataset("tatsu-lab/alpaca")


IF DATA SET IS SET TO "": SOLICIT COLUMN NAMES FROM ""

In [ ]:
dataset["train"].column_names


['instruction', 'input', 'output', 'text']

# Training Engine with definition for model, tokenizer, dataset, column of dataset, and length of training example

1/ Loads batches of text from your data set 2/ tokenizes them 3/ feeds them into the model 4/ computes the loss 5/ adjusts the model weights 6/ repeats this for the entire dataset


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./mistral-sft",
    fp16=True,
    bf16=False
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    args=training_args
)

trainer.tokenizer = tokenizer
trainer.train()


[2026-07-06 03:51:52,225] [WARNING] [real_accelerator.py:199:get_accelerator] Setting accelerator to CPU. If you have GPU or other accelerator, we were unable to detect it.


Adding EOS to train dataset:   0%|          | 0/52002 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/52002 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/52002 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/52002 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
import torch
import matplotlib.pyplot as plt

grads = []

for name, param in model.named_parameters():
    if param.grad is not None:
        grads.append(param.grad.abs().mean().item())

plt.plot(grads)
plt.title("Gradient Magnitudes Across Layers")
plt.xlabel("Layer Index")
plt.ylabel("Mean |Gradient|")
plt.show()


In [ ]:
for name, param in model.named_parameters():
    if param.grad is not None:
        print(name, param.grad.abs().mean().item())
        break
